<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/7_eleven_allonline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## To DO
'''
1. Change column name
2. using polars
3. Add column : Date, retailer
4. Extract Brand from Thai dict
5. import function re_evaluate_price from other source
6. add link dis
'''

'\n1. Change column name\n2. using polars\n3. Add column : Date, retailer\n4. Extract Brand from Thai dict\n5. import function re_evaluate_price from other source\n6. add link dis\n'

In [29]:
%%capture
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y ./google-chrome-stable_current_amd64.deb
!pip install selenium webdriver-manager beautifulsoup4 pandas xlsxwriter

In [3]:
from google.colab import data_table
data_table.enable_dataframe_formatter()


In [30]:
# from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
# import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
import xlsxwriter
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-16


In [17]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import polars as pl # Changed from pandas as pd

# 1. Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36")

print("Starting browser...")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# Base URL without the "p=" parameter
base_url = "https://allonline.7eleven.co.th/supermarket/household-items/laundry/liquid-detergent/?filter.BRAND=Fineline&filter.BRAND=Hygiene&filter.BRAND=%E0%B9%80%E0%B8%9B%E0%B8%B2&filter.BRAND=%E0%B9%81%E0%B8%AD%E0%B8%97%E0%B9%81%E0%B8%97%E0%B8%84&filter.PRICE=49-470&filter.initial_from_PRICE=49&filter.initial_to_PRICE=470&landing=true&pageSize=90&sortBy=si&view=0"

data = []
p_index = 0 # THE FIX: Start counting at 0!
previous_page_names = []

while True:
    # We display (p_index + 1) in the print statement just so it reads nicely for us as "Page 1", "Page 2"
    print(f"\nScraping Page {p_index + 1} (URL parameter p={p_index})...")

    # Inject the 0-based parameter
    current_page_url = base_url + f"&p={p_index}"
    driver.get(current_page_url)

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "item-description-cls-mobile"))
        )
    except Exception as e:
        print("No products found on this page. Reached the end!")
        break

    soup = BeautifulSoup(driver.page_source, "html.parser")
    product_titles = soup.find_all("div", class_="item-description-cls-mobile")

    if not product_titles:
        print("Page is empty. Reached the end!")
        break

    current_page_names = [title.text.strip() for title in product_titles]

    if current_page_names == previous_page_names:
        print("Duplicate items detected! We have reached the final page.")
        break

    previous_page_names = current_page_names.copy()

    # Parse Data
    for title_elem in product_titles:
        name = title_elem.text.strip()
        parent = title_elem.parent
        promotion_price_elem = None

        levels_climbed = 0
        while parent and parent.name != 'body' and levels_climbed < 5:
            promotion_price_elem = parent.find("strong")
            if promotion_price_elem:
                break
            parent = parent.parent
            levels_climbed += 1

        promotion_price = promotion_price_elem.text.strip() if promotion_price_elem else None
        original_price_elem = parent.find("s") if parent else None
        original_price = original_price_elem.text.strip() if original_price_elem else None

        data.append({
            "name": name,
            "promotion_price": promotion_price,
            "original_price": original_price
        })

    print(f"Successfully grabbed {len(product_titles)} items from Page {p_index + 1}.")

    # Increase the index for the next loop (0 becomes 1, 1 becomes 2)
    p_index += 1

driver.quit()

# Clean Data using Polars
df = pl.DataFrame(data)

if not df.is_empty():
    df = df.with_columns(
        pl.col("promotion_price").cast(pl.String).str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False).alias("promotion_price"),
        pl.col("original_price").cast(pl.String).str.replace_all(r"[^\d.]", "").cast(pl.Float64, strict=False).alias("original_price")
    )

    # Drop duplicates
    initial_count = df.height
    df = df.unique(subset=['name'], keep='first') # Polars unique handles duplicates similar to drop_duplicates(keep='first')

    print("\n--- Final Scraping Results ---")
    print(df.tail(10))
    print(f"\nFinal clean item count: {df.height}")

    filename = 'scoped_detergents.csv'
    df.write_csv(filename, separator=',', include_header=True) # Removed encoding='utf-8'
    print(f"\nData successfully saved to {filename}!")
else:
    print("No data was scraped.")

Starting browser...

Scraping Page 1 (URL parameter p=0)...
Successfully grabbed 90 items from Page 1.

Scraping Page 2 (URL parameter p=1)...
Successfully grabbed 13 items from Page 2.

Scraping Page 3 (URL parameter p=2)...
Duplicate items detected! We have reached the final page.

--- Final Scraping Results ---
shape: (10, 3)
┌───────────────────────────┬─────────────────┬────────────────┐
│ name                      ┆ promotion_price ┆ original_price │
│ ---                       ┆ ---             ┆ ---            │
│ str                       ┆ f64             ┆ f64            │
╞═══════════════════════════╪═════════════════╪════════════════╡
│ ไฟน์ไลน์พลัส น้ำยาซักผ้า สีทอง…  ┆ 169.0           ┆ 195.0          │
│ แอทแทค น้ำยาซักผ้า ไลฟ์ลี่ บลู…  ┆ 95.0            ┆ null           │
│ ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…  ┆ 271.0           ┆ null           │
│ ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…  ┆ 179.0           ┆ null           │
│ แอทแทค น้ำยาซักผ้ากลิ่นชาร์มมิ…  ┆ 219.0           ┆ 2

In [18]:
df

name,promotion_price,original_price
str,f64,f64
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันน…",179.0,null
"""ไฟน์ไลน์ น้ำยาซักผ้า สำหรับซัก…",271.0,null
"""เปา วินวอช ลิควิด พิ้งค์ซอฟท์ …",185.0,null
"""ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…",89.0,null
"""เปาวินวอช น้ำยาซักผ้า ลิควิด ส…",99.0,null
…,…,…
"""ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…",69.0,null
"""ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…",89.0,null
"""เปา วินวอชลิควิด ถุงเติม น้ำยา…",185.0,null


In [24]:
# @title udf data-prep
import polars as pl

def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original_prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# Apply the preparation function
df_prep_7_eleven = re_evaluate_price(df)

print("Data has been re-evaluated.")


Data has been re-evaluated.


In [8]:
df_prep_7_eleven

name,promotion_price,original_price
str,i64,f64
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันน…",238,271.0
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันน…",null,179.0
"""ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…",150,195.0
"""ไฟน์ไลน์ น้ำยาซักผ้า โปรคลีน ช…",150,195.0
"""ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…",150,195.0
…,…,…
"""น้ำยาซักผ้าแอทแทค เลดี้ อิลิแก…",null,95.0
"""แอทแทค น้ำยาซักผ้า ลัคชูรี่ โก…",null,95.0
"""แอทแทค น้ำยาซักผ้า บลู เอเมอรั…",null,95.0


In [36]:
import polars as pl
from datetime import date
import re

# Standardizing column names to match the requested function logic
# Assuming df_prep_7_eleven is the current working Polars dataframe

def parse_product_names_TH(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    Fixed to handle thousands separators (e.g., 1,300 ml).
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Updated patterns
    quant_unit_pattern = r"(?i)([\d,.]+)\s*(มล\.|ลิตร|ก\.ก\.|ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b|แพ็ก\s*\d+\s*ชิ้น)"

    # Define Thai brand extraction logic separately to maintain user's specific brand list
    thai_brands = ["ไฟน์ไลน์", "ไฮยีน", "เปา", "แอทแทค"]
    brand_pattern = r"^(" + "|".join(re.escape(b) for b in thai_brands) + r")"

    # 3. Apply the Polars transformation
    return df.with_columns(
        # Add condition column
        pl.lit(None).alias("condition"),

        pl.lit(today_date).alias("Date"),

        # Extract Brand: Check for Thai list first, then fallback to split
        pl.col("name").str.extract(brand_pattern, 1)
          .fill_null(pl.col("name").str.split(" ").list.first())
          .alias("Brand"),

        # Fixed Volume Extraction
        pl.col("name")
          .str.extract(quant_unit_pattern, 1)
          .str.replace_all(",", "")
          .cast(pl.Int64, strict=False)
          .alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer"),
    )

# Apply the transformation
df_trans_7_eleven = parse_product_names_TH(df_prep_7_eleven, "7-Eleven")

# Display the results with the new column order
(df_trans_7_eleven.select([
    "name",
    "promotion_price",
    "original_price",
    "condition",
    "Date",
    "Brand",
    "Volume",
    "Unit",
    "Pack",
    "Retailer"
    ]).head(15))

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,null,str,str,i64,str,str,str
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันน…",null,179.0,null,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven"""
"""ไฟน์ไลน์ น้ำยาซักผ้า สำหรับซัก…",null,271.0,null,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven"""
"""เปา วินวอช ลิควิด พิ้งค์ซอฟท์ …",null,185.0,null,"""2026-04-16""","""เปา""",null,null,null,"""7-Eleven"""
"""ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…",null,89.0,null,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven"""
"""เปาวินวอช น้ำยาซักผ้า ลิควิด ส…",null,99.0,null,"""2026-04-16""","""เปา""",null,null,null,"""7-Eleven"""
…,…,…,…,…,…,…,…,…,…
"""ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…",169.0,195.0,null,"""2026-04-16""","""ไฟน์ไลน์""",null,null,"""แพ็ก 3 ชิ้น""","""7-Eleven"""
"""น้ำยาซักผ้าแอทแทค คลีน แอดวานซ…",null,95.0,null,"""2026-04-16""","""น้ำยาซักผ้าแอทแทค""",null,null,null,"""7-Eleven"""
"""ไฟน์ไลน์ น้ำยาซักผ้า คูลลิ่งเฟ…",null,179.0,null,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven"""


In [37]:
df_trans_7_eleven

name,promotion_price,original_price,Date,Brand,Volume,Unit,Pack,Retailer,condition
str,f64,f64,str,str,i64,str,str,str,null
"""ไฟน์ไลน์ พลัส น้ำยาซักผ้า ซันน…",null,179.0,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven""",null
"""ไฟน์ไลน์ น้ำยาซักผ้า สำหรับซัก…",null,271.0,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven""",null
"""เปา วินวอช ลิควิด พิ้งค์ซอฟท์ …",null,185.0,"""2026-04-16""","""เปา""",null,null,null,"""7-Eleven""",null
"""ไฟน์ไลน์ น้ำยาซักผ้า สูตรเข้มข…",null,89.0,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven""",null
"""เปาวินวอช น้ำยาซักผ้า ลิควิด ส…",null,99.0,"""2026-04-16""","""เปา""",null,null,null,"""7-Eleven""",null
…,…,…,…,…,…,…,…,…,…
"""ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…",null,69.0,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven""",null
"""ไฟน์ไลน์ น้ำยาซักผ้า แฮปปี้เนส…",null,89.0,"""2026-04-16""","""ไฟน์ไลน์""",null,null,null,"""7-Eleven""",null
"""เปา วินวอชลิควิด ถุงเติม น้ำยา…",null,185.0,"""2026-04-16""","""เปา""",null,null,null,"""7-Eleven""",null


In [38]:
df_trans_7_eleven.write_excel(f"7_eleven_laundry_{today_date}.xlsx")